# Initial ML solution: evidence-use and source-diversity follow-up

This notebook reuses the completed **BM25 train/dev top-20 rankings** and frozen **BM25-evidence DistilBERT**. It does not repeat the 12-hour dense run. Experiments are: **A** frozen-model evidence blank/swap diagnostic; **B** deterministic distinct-source selection from BM25 top 20; **C** blinded passage audit export; **D** optional matched DistilBERT retraining on source-diverse train/dev evidence; and **E** optional task-specific weak URL-proxy reranking.

Run cells in order on a Colab GPU. Outputs go to a new Drive directory and expensive cells resume if interrupted. Model inference uses direct Python calls with progress bars, not a subprocess wrapper. The repository changes must be pushed to GitHub before running this notebook in Colab. Do **not** call blank/swap or source-URL-proxy results evidence of causal improvement; the audit is needed. Keep official test untouched.


In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice-next')
if REPO_DIR.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Colab checkout has local edits. Save them before pulling.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repo revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import csv, gc, json, random, runpy, time
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
HF_CACHE = Path('/content/huggingface_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, restart, then rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed/averitec'
TRAIN_BM25 = DATA_ROOT / 'outputs/bm25/train_bm25_top20_v2/train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
ORIGINAL_MODEL = DATA_ROOT / 'outputs/initial_ml_solution_rerun_bs16/transformer/bm25_top3'
NEXT = DATA_ROOT / 'outputs/initial_ml_solution_next_v1'
NEXT.mkdir(parents=True, exist_ok=True)
ABLATION = NEXT / 'evidence_ablation/frozen_bm25_distilbert'
DIVERSITY = NEXT / 'source_diversity'
DIVERSE_TRAIN = DIVERSITY / 'train_bm25_unique_url'
DIVERSE_DEV = DIVERSITY / 'dev_bm25_unique_url'
must_exist = [PROCESSED/'train.jsonl', PROCESSED/'dev.jsonl', TRAIN_BM25, DEV_BM25, DEV_QRELS, ORIGINAL_MODEL/'final_model/config.json', ORIGINAL_MODEL/'final_dev_predictions.jsonl', ORIGINAL_MODEL/'final_dev_metrics.json', ORIGINAL_MODEL/'transformer_run_config.json']
missing = [str(path) for path in must_exist if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing prior experiment artifacts:\n' + '\n'.join(missing))
for name in ('run_evidence_ablation.py', 'run_source_diversity.py', 'train_transformer_verifier.py', 'train_weak_url_reranker.py'):
    if not (REPO_DIR/'scripts'/name).is_file():
        raise FileNotFoundError(f'New script missing: {name}. Push local changes, then rerun setup/pull.')
print('Inputs ready; new output root:', NEXT)


In [ ]:
def complete(paths):
    for path in map(Path, paths):
        if not path.is_file() or path.stat().st_size == 0:
            return False
        if path.suffix == '.json':
            try:
                json.loads(path.read_text(encoding='utf-8'))
            except (OSError, ValueError):
                return False
    return True

def run_script(label, name, args, outputs):
    script = (REPO_DIR/'scripts'/name).resolve()
    if not script.is_relative_to(REPO_DIR.resolve()) or not script.is_file():
        raise FileNotFoundError(script)
    previous = sys.argv[:] 
    sys.argv = [str(script), *map(str, args)]
    print(f'\n=== {label} ===\nScript: {script}\nArguments: {args}', flush=True)
    start = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    except KeyboardInterrupt:
        print('Interrupted. Rerun this cell with the same settings to resume.', flush=True)
        raise
    finally:
        sys.argv = previous
        gc.collect()
        torch.cuda.empty_cache()
    if not complete(outputs):
        raise RuntimeError(f'{label} returned without complete artifacts: {outputs}')
    print(f'{label} finished in {(time.monotonic()-start)/60:.1f} minutes.')


## A. Does the frozen verifier respond to the passages?

Predict the same 500 dev claims using **no retrieved passages** and a deterministic **wrong-claim evidence swap**. The original model and its original predictions are unchanged. A missing candidate pool stays empty in the swap. This is a distribution-shift sensitivity test, not a fair new model or a causal estimate of retrieval quality. Check prediction-change counts alongside macro-F1 and safety errors.


In [ ]:
ablation_args = [
    '--claims', PROCESSED/'dev.jsonl', '--rankings', DEV_BM25,
    '--original-predictions', ORIGINAL_MODEL/'final_dev_predictions.jsonl',
    '--model-dir', ORIGINAL_MODEL/'final_model',
    '--model-run-config', ORIGINAL_MODEL/'transformer_run_config.json',
    '--output-dir', NEXT/'evidence_ablation', '--run-name', 'frozen_bm25_distilbert',
    '--batch-size', '16', '--checkpoint-every', '32', '--device', 'cuda', '--resume',
]
run_script('Frozen DistilBERT evidence-use diagnostic', 'run_evidence_ablation.py', ablation_args, [ABLATION/'ablation_report.json', ABLATION/'ablation_manifest.json', ABLATION/'blank_predictions.jsonl', ABLATION/'swapped_predictions.jsonl'])
ablation = json.loads((ABLATION/'ablation_report.json').read_text())
print('Original macro-F1:', ablation['original_metrics_recomputed']['macro_f1'])
for name, row in ablation['variants'].items():
    print(name, 'macro-F1:', row['metrics']['macro_f1'], 'changed predictions:', row['versus_original']['changed_prediction_count'], 'false reassurance:', row['metrics']['decision_error_rates']['false_reassurance_count'], 'false alarm:', row['metrics']['decision_error_rates']['false_alarm_count'])


## B. Source-diverse BM25 top three

Within each saved BM25 top 20, place the highest-ranked chunk from each normalized source URL before duplicate chunks. Missing URLs are treated as separate passages. No qrels or labels are used for ranking; dev qrels are read only for evaluation. The same rule is applied to train and dev to support an optional matched classifier run. This method cannot retrieve an absent source and may improve the URL proxy without improving actual evidence quality.


In [ ]:
run_script('Source diversity / train', 'run_source_diversity.py', [
    '--input-rankings', TRAIN_BM25, '--output-dir', DIVERSITY, '--run-name', 'train_bm25_unique_url', '--split', 'train', '--max-candidates', '20', '--metric-k', '3',
], [DIVERSE_TRAIN/'train_rankings.jsonl', DIVERSE_TRAIN/'train_metrics.json', DIVERSE_TRAIN/'train_manifest.json'])
run_script('Source diversity / dev', 'run_source_diversity.py', [
    '--input-rankings', DEV_BM25, '--qrels', DEV_QRELS, '--output-dir', DIVERSITY, '--run-name', 'dev_bm25_unique_url', '--split', 'dev', '--max-candidates', '20', '--metric-k', '3',
], [DIVERSE_DEV/'dev_rankings.jsonl', DIVERSE_DEV/'dev_metrics.json', DIVERSE_DEV/'dev_manifest.json'])
diversity_metrics = json.loads((DIVERSE_DEV/'dev_metrics.json').read_text())
print('BM25 URL-proxy hit@3:', diversity_metrics['original_bm25']['evidence_hit_at_3'])
print('Distinct-source URL-proxy hit@3:', diversity_metrics['diverse']['evidence_hit_at_3'])
print('Claims with changed top three:', diversity_metrics['changed_top3_claim_count'])


## C. Export a blinded passage audit

This cell creates a CSV with A/B top-three passages for up to 20 changed claims, including proxy gains and changed cases without a proxy gain. **Read the passage text** and fill `which_has_more_decisive_evidence` with `A`, `B`, `tie`, or `neither`. Do not read the separate key until after rating. The cell never overwrites an existing audit CSV, so your notes survive a Colab restart. URL-proxy judgments are used to sample cases and kept out of the rating CSV.


In [ ]:
from src.utils.io import read_jsonl, write_jsonl
from scripts.run_bm25 import load_qrels
from src.data.corpus import canonical_url
AUDIT_DIR = NEXT/'manual_audit'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_CSV = AUDIT_DIR/'source_diversity_blinded_cases.csv'
AUDIT_KEY = AUDIT_DIR/'source_diversity_key.jsonl'
if AUDIT_CSV.exists() or AUDIT_KEY.exists():
    if not (AUDIT_CSV.exists() and AUDIT_KEY.exists()):
        raise RuntimeError('Only one audit artifact exists; inspect manually before rerunning.')
    print('Audit files already exist; your ratings were not overwritten:', AUDIT_CSV)
else:
    qrels = {claim_id:set(ids) for claim_id,ids in load_qrels(DEV_QRELS).items()}
    old = {row['claim_id']:row for row in read_jsonl(DEV_BM25)}
    new = {row['claim_id']:row for row in read_jsonl(DIVERSE_DEV/'dev_rankings.jsonl')}
    if set(old) != set(new):
        raise ValueError('BM25 and source-diverse dev claim sets differ')
    categories = {'proxy_gain':[], 'changed_no_gain':[]}
    for claim_id in sorted(old):
        old_top = old[claim_id]['retrieved'][:3]
        new_top = new[claim_id]['retrieved'][:3]
        if [item['passage_id'] for item in old_top] == [item['passage_id'] for item in new_top]:
            continue
        judged = claim_id in qrels
        old_hit = judged and any(item['passage_id'] in qrels[claim_id] for item in old_top)
        new_hit = judged and any(item['passage_id'] in qrels[claim_id] for item in new_top)
        category = 'proxy_gain' if new_hit and not old_hit else 'changed_no_gain'
        categories[category].append((claim_id, old_hit, new_hit))
    rng = random.Random(718)
    for cases in categories.values():
        rng.shuffle(cases)
    selected = categories['proxy_gain'][:10] + categories['changed_no_gain'][:10]
    rng.shuffle(selected)
    fields = ['claim_id','claim','A_top3','B_top3','which_has_more_decisive_evidence','notes']
    key_rows = []
    temporary = AUDIT_CSV.with_name(AUDIT_CSV.name+'.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for claim_id, old_hit, new_hit in selected:
            old_is_A = bool(rng.getrandbits(1))
            A = old[claim_id]['retrieved'][:3] if old_is_A else new[claim_id]['retrieved'][:3]
            B = new[claim_id]['retrieved'][:3] if old_is_A else old[claim_id]['retrieved'][:3]
            def format_top(items):
                return '\n\n'.join(f"{index}. [{item['passage_id']}] {canonical_url(item.get('url')) or 'unknown URL'}\n{item.get('text','')}" for index,item in enumerate(items,1))
            writer.writerow({'claim_id':claim_id,'claim':old[claim_id]['claim'],'A_top3':format_top(A),'B_top3':format_top(B),'which_has_more_decisive_evidence':'','notes':''})
            key_rows.append({'claim_id':claim_id,'A_system':'bm25' if old_is_A else 'source_diverse','B_system':'source_diverse' if old_is_A else 'bm25','bm25_proxy_hit':old_hit,'diverse_proxy_hit':new_hit})
    temporary.replace(AUDIT_CSV)
    write_jsonl(key_rows, AUDIT_KEY)
    print('Created', len(selected), 'blinded audit cases:', AUDIT_CSV)
    print('Keep the answer key closed until ratings are finished:', AUDIT_KEY)


### Score the audit only after rating every case

Fill all ratings in the CSV first, then set `REVEAL_AUDIT = True`. The hidden key is read only after that flag is changed. `tie` means both sets contain similarly decisive evidence; `neither` means neither is decisive. This small, enriched sample is diagnostic rather than a representative estimate of retrieval accuracy.


In [ ]:
REVEAL_AUDIT = False  # Fill the entire CSV before changing this.
if REVEAL_AUDIT:
    with AUDIT_CSV.open(encoding='utf-8', newline='') as handle:
        rated = list(csv.DictReader(handle))
    allowed = {'A', 'B', 'tie', 'neither'}
    invalid = [(row['claim_id'], row['which_has_more_decisive_evidence']) for row in rated if row['which_has_more_decisive_evidence'].strip() not in allowed]
    if invalid:
        raise ValueError(f'Complete all audit ratings first; missing/invalid values: {invalid[:5]}')
    key = {row['claim_id']:row for row in read_jsonl(AUDIT_KEY)}
    if len(key) != len(rated) or {row['claim_id'] for row in rated} != set(key):
        raise ValueError('Rated CSV and hidden key have different claim IDs')
    counts = {'bm25':0, 'source_diverse':0, 'tie':0, 'neither':0}
    proxy_gain_counts = counts.copy()
    for row in rated:
        answer = row['which_has_more_decisive_evidence'].strip()
        winner = key[row['claim_id']][answer+'_system'] if answer in {'A','B'} else answer
        counts[winner] += 1
        if key[row['claim_id']]['diverse_proxy_hit'] and not key[row['claim_id']]['bm25_proxy_hit']:
            proxy_gain_counts[winner] += 1
    print('Manual evidence preference across changed cases:', counts)
    print('Within URL-proxy-gain cases:', proxy_gain_counts)
else:
    print('Audit key remains sealed. Fill the CSV before revealing results.')


## D. Optional: retrain DistilBERT with source-diverse evidence

Only run after inspecting the manual audit. Set `RUN_MATCHED_TRAINING = True` to fit the same four learning-rate/epoch trials on **source-diverse train evidence** and evaluate on **source-diverse dev evidence**. Training resumes from Drive checkpoints. The model is selected on the train-only 307-claim validation fold, not by dev performance. If the audit shows no genuine gain, leave this off and report the negative retrieval result.


In [ ]:
RUN_MATCHED_TRAINING = False  # Change to True only after reviewing the audit.
DIVERSE_MODEL = NEXT/'transformer/bm25_source_diverse_top3'
if RUN_MATCHED_TRAINING:
    prior_config = json.loads((ORIGINAL_MODEL/'transformer_run_config.json').read_text())
    revision = prior_config.get('resolved_model_revision') or prior_config.get('requested_revision')
    if not revision:
        raise ValueError('Original DistilBERT revision was not recorded; inspect before training.')
    train_args = [
        '--data-root', DATA_ROOT, '--train-rankings', DIVERSE_TRAIN/'train_rankings.jsonl',
        '--eval-rankings', DIVERSE_DEV/'dev_rankings.jsonl',
        '--output-dir', NEXT/'transformer', '--run-name', 'bm25_source_diverse_top3',
        '--model', prior_config['model'], '--revision', revision,
        '--max-length', str(prior_config['max_length']),
        '--claim-token-budget', str(prior_config['claim_token_budget']),
        '--evidence-token-budget', str(prior_config['evidence_token_budget_per_passage']),
        '--validation-fraction', str(prior_config['validation_fraction']),
        '--learning-rates', ','.join(map(str, prior_config['learning_rate_grid'])),
        '--epoch-grid', ','.join(map(str, prior_config['epoch_grid'])),
        '--batch-size', str(prior_config['batch_size']),
        '--gradient-accumulation-steps', str(prior_config['gradient_accumulation_steps']),
        '--checkpoint-steps', str(prior_config['checkpoint_steps']),
        '--seed', str(prior_config['validation_seed']), '--resume',
    ]
    run_script('DistilBERT / source-diverse BM25', 'train_transformer_verifier.py', train_args, [DIVERSE_MODEL/'final_dev_metrics.json', DIVERSE_MODEL/'final_dev_predictions.jsonl', DIVERSE_MODEL/'transformer_manifest.json'])
    result = json.loads((DIVERSE_MODEL/'final_dev_metrics.json').read_text())
    print('Source-diverse model macro-F1:', result['macro_f1'])
    print('False reassurance:', result['decision_error_rates']['false_reassurance_count'], 'false alarm:', result['decision_error_rates']['false_alarm_count'])
else:
    print('Matched classifier training skipped. Complete the manual audit first.')


## E. Optional: task-specific weak URL-proxy reranker

Only run this after judging whether the URL proxy tracks decisive evidence in the manual audit. Train a logistic reranker using **train-only annotated source URLs** as weak passage labels and claim/passage lexical features plus BM25 rank/score. A held-out subset of **train claims** selects regularization and score blending; official dev labels are read only after the final ranking is frozen. This is a retrieval experiment, **not** an end-to-end verdict result. Do not train a verifier on its in-sample train rerankings: that needs out-of-fold train rankings to avoid relevance-label leakage. The run is short and idempotent; rerun the cell if interrupted.


In [ ]:
RUN_WEAK_RERANKER = False  # Change to True only after reviewing the manual audit.
WEAK = NEXT/'weak_url_reranker/train_proxy_v1'
if RUN_WEAK_RERANKER:
    run_script('Train weak URL-proxy reranker', 'train_weak_url_reranker.py', [
        '--train-claims', PROCESSED/'train.jsonl', '--train-rankings', TRAIN_BM25,
        '--dev-rankings', DEV_BM25, '--dev-qrels', DEV_QRELS,
        '--output-dir', NEXT/'weak_url_reranker', '--run-name', 'train_proxy_v1',
        '--max-candidates', '20', '--seed', '718', '--validation-fraction', '0.10',
    ], [WEAK/'model.joblib', WEAK/'model_selection.json', WEAK/'dev_rankings.jsonl', WEAK/'dev_metrics.json', WEAK/'weak_reranker_manifest.json'])
    selection = json.loads((WEAK/'model_selection.json').read_text())
    report = json.loads((WEAK/'dev_metrics.json').read_text())
    print('Train-only validation selected:', selection['selected'])
    print('BM25 URL-proxy hit@3:', report['bm25_reference']['evidence_hit_at_3'])
    print('Weak reranker URL-proxy hit@3:', report['dev']['evidence_hit_at_3'])
else:
    print('Weak reranker skipped. Validate the URL proxy with the manual audit first.')


## Readout and decision gate

Compare source-diverse retrieval with original BM25 at k=3 and read the blinded audit before claiming improvement. If the frozen verifier barely reacts to blanked/swapped evidence, work on evidence use before a more expensive retriever. If distinct-source selection genuinely improves decisive passages, compare its matched DistilBERT against the existing BM25 and reranked models on all 500 dev claims, especially false reassurance. The optional weak reranker is selected without dev labels, but its URL-proxy score remains a diagnostic until manual passage judgments support it. Multi-query BM25 is a later experiment.


In [ ]:
baseline = json.loads((ORIGINAL_MODEL/'final_dev_metrics.json').read_text())
print('Original BM25 + DistilBERT:', {'macro_f1':baseline['macro_f1'], 'accuracy':baseline['accuracy'], 'safety':baseline['decision_error_rates']})
if (DIVERSE_DEV/'dev_metrics.json').is_file():
    m = json.loads((DIVERSE_DEV/'dev_metrics.json').read_text())
    print('BM25 vs source-diverse URL-proxy hit@3:', m['original_bm25']['evidence_hit_at_3'], m['diverse']['evidence_hit_at_3'])
if (DIVERSE_MODEL/'final_dev_metrics.json').is_file():
    m = json.loads((DIVERSE_MODEL/'final_dev_metrics.json').read_text())
    print('Source-diverse BM25 + DistilBERT:', {'macro_f1':m['macro_f1'], 'accuracy':m['accuracy'], 'safety':m['decision_error_rates']})
if (WEAK/'dev_metrics.json').is_file():
    m = json.loads((WEAK/'dev_metrics.json').read_text())
    print('Weak URL-proxy reranker retrieval:', {'bm25_hit_at_3':m['bm25_reference']['evidence_hit_at_3'], 'reranker_hit_at_3':m['dev']['evidence_hit_at_3'], 'judged_claim_count':m['dev']['judged_claim_count']})
print('All new artifacts:', NEXT)
